In [4]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv("/content/project_data_cleaned_v1.csv")
data.head()

,comment_id,text,published_at,roberta_neg,roberta_neu,roberta_pos,roberta_label,num_words
0,UgwoiJ6zDlEHgNHbKHl4AaABAg,mike is the most traumatic in this season his ...,2025-12-27T02:17:44Z,0.971900,0.025935,0.002164,negative,42
1,UgyXJRcSbn74ajhkmBN4AaABAg,"this was so bad yall. wth was that writing, th...",2025-12-27T02:13:56Z,0.975026,0.022050,0.002924,negative,54
2,UgzhXrqNFCkMLMOB6dh4AaABAg,not a single shot from final episode,2025-12-27T01:51:33Z,0.390147,0.579394,0.030459,neutral,7
3,UgxQPzUbRC8ECcXroFF4AaABAg,rated j for jesus: a box office blasphemy – ho...,2025-12-27T01:35:21Z,0.299472,0.592630,0.107898,neutral,15
4,UgzQIyIMuD3WZha30dd4AaABAg,does hollyweird ever stop attacking christiani...,2025-12-27T01:30:37Z,0.972716,0.024830,0.002454,negative,27


In [3]:
data.shape

(19596, 8)

In [5]:
# Ensure necessary NLTK data is downloaded
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [8]:
# Define the preprocessing function
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment

In [10]:
from tqdm import tqdm
tqdm.pandas()

data["text"] = data["text"].progress_apply(preprocess_comment)

100%|██████████| 19596/19596 [00:03<00:00, 5205.47it/s]


In [17]:
# juswt keeping the text and the sentiment
data = data.loc[:,["text", "roberta_label"]] \
      .rename(columns={"roberta_label" : "label"})

In [13]:
def encode_label(string):
  """takes a sentiment label and converts it to 0 if neutral, 1 if positive and -1 if negative"""
  if string == "positive":
    return 1
  elif string == "negative":
    return -1
  else:
    return 0

data["label"] = data["label"].progress_apply(encode_label)

100%|██████████| 19596/19596 [00:00<00:00, 957876.85it/s]


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Example data
X = data["text"]
y = data["label"]

# 1️⃣ Split first
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2️⃣ Fit ONLY on training data
vectorizer = CountVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)

# 3️⃣ Transform test data
X_test_vec = vectorizer.transform(X_test)

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

n_trees = 200

rf = RandomForestClassifier(
    n_estimators=1,
    warm_start=True,   # 🔑 allows incremental training
    random_state=42,
    n_jobs=-1
)

for i in tqdm(range(1, n_trees + 1), desc="Training Random Forest"):
    rf.n_estimators = i
    rf.fit(X_train_vec, y_train)

Training Random Forest: 100%|██████████| 200/200 [02:07<00:00,  1.56it/s]


In [21]:
y_pred = rf.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6686224489795919
              precision    recall  f1-score   support

          -1       0.76      0.75      0.75      1483
           0       0.62      0.58      0.60      1395
           1       0.61      0.68      0.64      1042

    accuracy                           0.67      3920
   macro avg       0.66      0.67      0.66      3920
weighted avg       0.67      0.67      0.67      3920



In [25]:
# trying tfidf
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3
)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2️⃣ Fit ONLY on training data
X_train_vec = vectorizer.fit_transform(X_train)

# 3️⃣ Transform test data
X_test_vec = vectorizer.transform(X_test)

In [27]:
n_trees = 200

rf = RandomForestClassifier(
    n_estimators=1,
    warm_start=True,   # 🔑 allows incremental training
    random_state=42,
    n_jobs=-1
)

for i in tqdm(range(1, n_trees + 1), desc="Training Random Forest"):
    rf.n_estimators = i
    rf.fit(X_train_vec, y_train)

Training Random Forest: 100%|██████████| 200/200 [01:30<00:00,  2.21it/s]


In [28]:
y_pred = rf.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6936224489795918
              precision    recall  f1-score   support

          -1       0.75      0.78      0.76      1483
           0       0.62      0.68      0.65      1395
           1       0.73      0.60      0.66      1042

    accuracy                           0.69      3920
   macro avg       0.70      0.68      0.69      3920
weighted avg       0.70      0.69      0.69      3920



In [29]:
test_comments = [
    "This product completely exceeded my expectations, I love it",
    "It’s okay I guess, nothing special",
    "Worst experience ever, total waste of money",
    "The delivery was on time and the packaging was fine",
    "I regret buying this",
    "Works fine so far, no major issues",
    "Absolutely terrible customer service",
    "Pretty decent overall",
    "I feel neutral about this product",
    "Amazing experience, would recommend to everyone"
]

In [30]:
label_map = {
    -1: "Negative",
     0: "Neutral",
     1: "Positive"
}

for comment in test_comments:
    comment_vec = vectorizer.transform([comment])
    pred = rf.predict(comment_vec)[0]

    print(f"Comment: {comment}")
    print(f"Predicted sentiment: {label_map[pred]}")
    print("-" * 50)

Comment: This product completely exceeded my expectations, I love it
Predicted sentiment: Positive
--------------------------------------------------
Comment: It’s okay I guess, nothing special
Predicted sentiment: Negative
--------------------------------------------------
Comment: Worst experience ever, total waste of money
Predicted sentiment: Negative
--------------------------------------------------
Comment: The delivery was on time and the packaging was fine
Predicted sentiment: Neutral
--------------------------------------------------
Comment: I regret buying this
Predicted sentiment: Neutral
--------------------------------------------------
Comment: Works fine so far, no major issues
Predicted sentiment: Negative
--------------------------------------------------
Comment: Absolutely terrible customer service
Predicted sentiment: Positive
--------------------------------------------------
Comment: Pretty decent overall
Predicted sentiment: Neutral
----------------------------